In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "SOLUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,200.62,200.62,200.19,200.43,8099.652,2025-09-01 00:00:59.999999+00:00,1.623070e+06,3822,2579.174,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,200.44,200.57,200.36,200.57,2420.952,2025-09-01 00:01:59.999999+00:00,4.853247e+05,1832,967.795,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.003141,0.001745,0.001396,NaN,NaN
2,2025-09-01 00:02:00+00:00,200.57,200.58,200.21,200.36,2998.765,2025-09-01 00:02:59.999999+00:00,6.007899e+05,2143,1127.508,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.002510,0.000001,-0.002511,NaN,NaN
3,2025-09-01 00:03:00+00:00,200.37,200.44,200.24,200.24,1907.570,2025-09-01 00:03:59.999999+00:00,3.821583e+05,1852,766.376,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.009351,-0.003167,-0.006184,NaN,NaN
4,2025-09-01 00:04:00+00:00,200.25,200.25,199.65,199.66,32397.094,2025-09-01 00:04:59.999999+00:00,6.479208e+06,6276,3251.055,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.035951,-0.012919,-0.023032,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 05:33:08,253] A new study created in memory with name: no-name-86c92de4-f8de-4c6a-9f28-990b129a7976


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:05<?, ?it/s]

Best trial: 0. Best value: 0.0201592:   0%|          | 0/50 [00:05<?, ?it/s]

Best trial: 0. Best value: 0.0201592:   2%|▏         | 1/50 [00:05<04:10,  5.10s/it]

...

Best trial: 47. Best value: 0.0420694:  98%|█████████▊| 49/50 [02:50<00:02,  2.54s/it]

Best trial: 47. Best value: 0.0420694:  98%|█████████▊| 49/50 [02:50<00:02,  2.54s/it]

Best trial: 47. Best value: 0.0420694: 100%|██████████| 50/50 [02:50<00:00,  2.50s/it]

Best trial: 47. Best value: 0.0420694: 100%|██████████| 50/50 [02:50<00:00,  3.40s/it]

[I 2026-03-20 05:35:58,382] Trial 49 finished with value: 0.035417070393105585 and parameters: {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.0013348179132139475, 'subsample': 0.8687737167206874, 'colsample_bytree': 0.9110209213206192, 'min_child_weight': 5, 'reg_alpha': 1.1704504621680254, 'reg_lambda': 0.15834312665720893}. Best is trial 47 with value: 0.04206943180936533.

[optuna] best trial
value: 0.042069
params:
  n_estimators: 800
  max_depth: 4
  learning_rate: 0.0011732568667010754
  subsample: 0.6006030983456705
  colsample_bytree: 0.9367941262470788
  min_child_weight: 5
  reg_alpha: 2.355503697047383
  reg_lambda: 0.011950366753912315


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 4.58s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...



===== RESULTS =====
Train IC:      0.034202
Test IC:       0.014493
Train Rank IC: 0.046635
Test Rank IC:  0.035137
Train RMSE:    0.002480
Test RMSE:     0.002517


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
mom_5               0.091138
vol_15              0.062831
mom_3               0.058236
hour_sin            0.043216
dom_sin             0.040112
dow_cos             0.036871
dow_sin             0.034844
atr_norm            0.034346
is_high_vol         0.031053
macd_hist           0.028714
range_15            0.028214
mom_x_imb           0.027916
vol_30              0.026078
month_sin           0.025321
vol_regime_ratio    0.025152
vol_ratio_5_30      0.024673
dist_ma_5           0.023317
trend_strength      0.022136
vol_5               0.021486
range_ratio         0.021422
imbalance_15        0.020133
mr_x_vol            0.019832
mom_60              0.019471
trades_z            0.018311
imbalance           0.018111
trend_x_imb         0.017963
dist_ma_15_z        0.017735
dom_cos             0.016460
volume_mom_5        0.014973
mom_30              0.013445
bar_range           0.013147
dist_ma_15          0.011970
dist_ma_30          0.011593
month_cos  

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/SOLUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/SOLUSDT__h5_model.joblib
[saved] features -> models/xgb/SOLUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/SOLUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/SOLUSDT__h5_meta.json
